# GÜN 087: exec() Sistem Çağrısı Ailesinin Zihinsel Modeli, Adres Uzayı Dönüşümü ve POSIX Süreç Mutasyonu

```
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR

Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)

Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim
amaçlı olarak paylaşılmıştır.

YASAKLAR:
  1. Kopyalanamaz, çoğaltılamaz, dağıtılamaz veya yeniden yayınlanamaz.
  2. Ticari veya ticari olmayan hiçbir projede kullanılamaz, değiştirilemez.
  3. Alt lisanslanamaz, satılamaz veya devredilemez.
  4. Tersine mühendislik yapılamaz.

İZİN VERİLEN KULLANIM:
  - GitHub üzerinde görüntüleme ve okuma.
  - Kişisel öğrenim amacıyla kodu inceleme (kopyalamadan).

YAZARIN AÇIK YAZILI İZNİ OLMAKSIZIN HİÇBİR KULLANIM HAKKI TANINMAZ.
İzin talepleri için: GitHub @seydivakkas
```

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)

---

## Müfredat ve Savunma Sanayii Seviyesi Bilgilendirmesi

- **Faz**: F2 — Linux / POSIX Sistem Programlama ve Donanım Yakın Mimari
- **Konu**: `exec` Ailesi Zihinsel Modeli (The `exec` Family Mental Model & Address Space Mutation)
- **Ön Koşul Bilgi**: Gün 086 (`fork` zihinsel modeli, Copy-on-Write, sayfa tablosu klonlama, `atfork` işleyicileri)
- **Hedef Sistemler**: ASELSAN Görev Bilgisayarları, HAVELSAN Komuta Kontrol (C2) Süreç Yöneticileri, TUSAŞ Uçuş Yönetim Sistemleri (FMS), Gerçek Zamanlı Linux (RT-PREEMPT) ve POSIX Uyumlu Gömülü Dağıtımlar.

### 40-Nokta Kalite ve Mimari Beyanı
Bu çalışma paketi; POSIX standardı (`IEEE Std 1003.1-2024`), Linux Kernel v6.x bellek yönetim altsistemi (`mm/mmap.c`, `fs/binfmt_elf.c`) ve askeri görev kritik sistem güvenlik yönergeleri (MISRA C++20, DO-178C Level A) gözetilerek üretilmiştir. İçerik; salt teorik tanımların ötesinde, x86_64 sanal adres uzayı dönüşüm mekanizmalarını, 7 farklı libc fonksiyonunun tekil `execve(2)` sistem çağrısına indirgenme topolojisini, dosya tanıtıcısı sızıntılarını (`FD_CLOEXEC` / `O_CLOEXEC`), atomik alt süreç başlatma alternatifi olan `posix_spawn(3)` optimizasyonunu ve katı matematiksel bellek sınırlarını (`ARG_MAX`, $T_{\text{exec}}$ gecikme modeli) derinlemesine ele alır.


## 1. Teori ve Sistem Mimarisi: Sanal Bellek Uzayının Dönüşümü

İşletim sistemleri teorisinde `fork()` bir sürecin genetik ikizini üretirken, `exec()` ailesi bu bedenin içerisine yepyeni bir "ruh" üfler. En yaygın yanılgı, `exec` çağrısının yeni bir süreç başlattığı yönündedir. **`exec` yeni bir süreç oluşturmaz; çalışan sürecin kimliğini (PID) koruyarak bellek uzayını, kodunu ve çalışma durumunu baştan aşağı değiştirir.**

```
       +-------------------------------------------------------------+
       |                  MEVCUT SÜREÇ (PID: 4092)                    |
       |  +-------------------------------------------------------+  |
       |  |  Sanal Adres Uzayı (VMA): Eski Binary (.text, .data)   |  |
       |  |  Heap, Yığın (Stack), TLS, Anonim Sayfalar            |  |
       |  +-------------------------------------------------------+  |
       +-------------------------------------------------------------+
                                      |
                     SYSCALL: execve("/bin/target", argv, envp)
                                      |
                                      v
       +-------------------------------------------------------------+
       |                  LINUX ÇEKİRDEK OPERASYONLARI               |
       |  1. binfmt_elf: ELF başlığı (magic: 0x7F 'E' 'L' 'F') oku  |
       |  2. munmap(): Eski sanal adres uzayının TÜMÜNÜ unmap et    |
       |  3. Yeni VMA'lar oluştur: .text (RX), .data (RW), .bss     |
       |  4. Dinamik Yükleyiciyi Haritala: ld-linux-x86-64.so.2     |
       |  5. Stack sıfırla; argv, envp ve Auxv vektörünü en üste yaz |
       |  6. Kayıtçıları (Registers: RIP, RSP) yeni giriş noktasına |
       |     (entry point: _start) ayarla.                           |
       +-------------------------------------------------------------+
                                      |
                                      v
       +-------------------------------------------------------------+
       |                MUTASYONA UĞRAMIŞ SÜREÇ (PID: 4092)          |
       |  +-------------------------------------------------------+  |
       |  |  Sanal Adres Uzayı: /bin/target İkili Kodu (.text)     |  |
       |  |  Yeni Boş Heap (brk), Sıfırlanmış Stack, Yeni TLS     |  |
       |  +-------------------------------------------------------+  |
       +-------------------------------------------------------------+
```

### POSIX `exec` Ailesi Taksonomisi (7 Wrapper, 1 Çekirdek Sistem Çağrısı)

Linux C kütüphanesinde (`glibc`, `musl`) bulunan 7 farklı `exec` fonksiyonu, geliştirici ergonomisi için sunulmuş olup nihayetinde tek bir çekirdek sistem çağrısına indirgenir: **`sys_execve` (x86_64 Syscall #59)**.

| Fonksiyon Adı | Argüman Formatı | PATH Araması Yapılır mı? | Ortam Değişkeni (`envp`) Kaynağı | Syscall Karşılığı |
| :--- | :--- | :--- | :--- | :--- |
| `execl()` | Liste (`char *arg0, ...`) | Hayır (Mutlak/Göreceli Yol) | Çağıran sürecin `environ` | `execve` |
| `execlp()` | Liste (`char *arg0, ...`) | **Evet (`$PATH` taranır)** | Çağıran sürecin `environ` | `execve` |
| `execle()` | Liste (`char *arg0, ...`) | Hayır (Mutlak/Göreceli Yol) | **Açıkça Verilen `char *envp[]`** | `execve` |
| `execv()` | Dizi/Vektör (`char *argv[]`) | Hayır (Mutlak/Göreceli Yol) | Çağıran sürecin `environ` | `execve` |
| `execvp()` | Dizi/Vektör (`char *argv[]`) | **Evet (`$PATH` taranır)** | Çağıran sürecin `environ` | `execve` |
| `execvpe()` | Dizi/Vektör (`char *argv[]`) | **Evet (`$PATH` taranır)** | **Açıkça Verilen `char *envp[]`** | `execve` |
| **`execve()`** | **Dizi/Vektör (`char *argv[]`)** | **Hayır (Doğrudan Yol)** | **Açıkça Verilen `char *envp[]`** | **Doğrudan Çekirdek Syscall (#59)** |

> [!IMPORTANT]
> İsimlendirme Formülü:
> - **`l` (List)**: Argümanlar variadic liste olarak (`NULL` ile sonlandırılmış) geçirilir.
> - **`v` (Vector)**: Argümanlar `NULL` sonlandırıcılı bir `char*` işaretçi dizisi olarak geçirilir.
> - **`p` (Path)**: Dosya adı slash (`/`) içermiyorsa, sistem `$PATH` ortam değişkenindeki dizinleri sırayla arar.
> - **`e` (Environment)**: Süreç varsayılan `environ` tablosunu devralmaz; açıkça bir `char *const envp[]` dizisi geçirilir (Savunma sistemlerinde ortam izolasyonu için zorunludur).


## 2. POSIX Süreç Durumu: Neler Korunur, Neler Sıfırlanır?

Bir sistem mühendisi için `execve` çağrısının neyi yok edip neyi muhafaza ettiğini bilmek hayati önem taşır. Bu ayrım, güvenlik zafiyetlerinin ve yarış durumlarının temel kaynağıdır.

### Korunan ve Sıfırlanan Nitelikler Matrisi

```
+------------------------------------+------------------------------------+
| KORUNAN NİTELİKLER (Persisted)     | SIFIRLANAN NİTELİKLER (Reset)      |
+------------------------------------+------------------------------------+
| 1. PID, PPID, PGID, SID            | 1. Sanal Adres Uzayı (VMA / Text)  |
| 2. Real UID / Real GID             | 2. Heap (brk/mmap tahsisleri)      |
| 3. Supplementary Group IDs         | 3. Çağrı Yığını (Call Stack)       |
| 4. Açık Dosya Tanıtıcıları (FDs)   | 4. Tüm Thread'ler (Main hariç yok) |
|    (Yalnızca FD_CLOEXEC OLMAYANLAR)| 5. Sinyal İşleyicileri (SIG_DFL)   |
| 5. Dosya Ofsetleri ve Kilitleri    | 6. FD_CLOEXEC bayraklı dosya tan.  |
| 6. Bekleyen Sinyaller & Maskesi    | 7. Bellek Kilitleri (mlock/mlockall)|
| 7. Kaynak Limitleri (RLIMIT_*)     | 8. AIO (Asenkron I/O) Blokları     |
| 8. CPU Zamanlayıcı Önceliği (nice) | 9. Thread-Local Storage (TLS)      |
| 9. Çalışma Dizini (cwd) ve umask   | 10. Program Sayacı (RIP/PC)        |
+------------------------------------+------------------------------------+
```

### Sinyal Davranışı Özelliği: SIG_DFL vs SIG_IGN
- **Özel Fonksiyon İşleyicileri (Custom Handlers)**: Eski sanal bellek uzayı unmap edildiği için, kullanıcı koduna işaret eden fonksiyon adresleri anlamsız hale gelir. Çekirdek, özel tanımlı tüm sinyal işleyicilerini derhal `SIG_DFL` (varsayılan aksiyon) moduna sıfırlar.
- **Yoksayılan Sinyaller (`SIG_IGN`)**: Eğer bir sinyal ebeveyn süreç tarafından yoksayılmışsa (`SIG_IGN`), bu durum `execve` sonrasında da **KORUNUR**. Örneğin arka planda çalışan daemon'larda `signal(SIGINT, SIG_IGN)` ayarlanmışsa, çağrılan alt ikili dosya da `SIGINT` sinyallerini yoksaymaya devam eder.
- **Sinyal Maskesi (`sigprocmask`)**: Bloke edilen sinyal kümesi `execve` boyunca aynen korunur.


## 3. Güvenlik ve İzolasyon: Dosya Tanıtıcısı Sızıntısı (FD Leak)

Askeri aviyonik ve dağıtık komuta kontrol sistemlerinde en ölümcül güvenlik zafiyetlerinden biri **Dosya Tanıtıcısı Sızıntısıdır (File Descriptor Leakage)**.

### Sızıntı Mekanizması ve Saldırı Yüzeyi
Bir ana süreç (Örn: Radar Görev Yöneticisi), donanım erişimi veya kriptografik anahtarlar için bir dosya tanıtıcısı açar (FD: 3). Ardından sistem teşhis ve loglama amacıyla harici bir alt süreç başlatır (`fork()` + `execve("/usr/bin/diag_tool")`).
Eğer FD 3 üzerinde **`FD_CLOEXEC`** bayrağı ayarlanmamışsa:
1. `execve` çağrısı sanal belleği sıfırlasa bile, açık dosya tanıtıcıları tablosunu (`files_struct`) **YENİ PROGRAMA AKTARIR**.
2. `/usr/bin/diag_tool` süreci kendi kodunda `open()` çağırmamış olsa bile, `read(3, buffer, ...)` veya `write(3, ...)` yaparak radar kripto anahtarlarına veya donanım DMA kontrol soketlerine doğrudan erişebilir!
3. Yetki yükseltme (Privilege Escalation) ve veri sızıntısı (Information Disclosure) kaçınılmaz hale gelir.

```
Ana Süreç (Parent)                execve("/bin/diag")              Alt Süreç (Mutated Child)
+-----------------------+                                         +-----------------------+
| FD 0: stdin           | --------------------------------------> | FD 0: stdin           |
| FD 1: stdout          | --------------------------------------> | FD 1: stdout          |
| FD 2: stderr          | --------------------------------------> | FD 2: stderr          |
| FD 3: CRYPTO_KEY.PEM  | === FD_CLOEXEC YOK (SIZINTI!) =======> | FD 3: CRYPTO_KEY.PEM  |
| FD 4: RADAR_SOCKET    | --- FD_CLOEXEC VAR (GÜVENLİ KAPATMA) -> | [KAPATILDI - EBADF]   |
+-----------------------+                                         +-----------------------+
```

### Yarış Koşulu (Race Condition) ve `O_CLOEXEC` Zorunluluğu
Geleneksel POSIX kodlarında dosya açıldıktan sonra `fcntl()` ile bayrak atanıyordu:
```c
int fd = open("/secret/crypto.key", O_RDONLY);
// <-- YARIŞ KOŞULU: Başka bir thread tam bu milisaniyede fork() + execve() çağırırsa FD sızar!
fcntl(fd, F_SETFD, FD_CLOEXEC);
```
Modern POSIX (Linux 2.6.27+) bu ölümcül TOCTOU açığını kapatmak için **atomik `O_CLOEXEC`** bayrağını zorunlu kılmıştır:
```c
int fd = open("/secret/crypto.key", O_RDONLY | O_CLOEXEC); // Atomik ve sızıntısız!
```


## 4. Modern Süreç Başlatma Mimarisi: `posix_spawn` vs `fork() + execve()`

Geleneksel Unix modeli `fork() + execve()` ikilisini kullanır. Ancak modern yüksek bellekli görev kritik sistemlerde bu model ciddi performans ve kararlılık darboğazlarına yol açar.

### Neden `fork() + execve()` Hantal Kalabilir?
1. **Sayfa Tablosu Klonlama Yükü**: `fork()` Copy-on-Write (CoW) kullansa bile, ebeveyn sürecin tüm sayfa tablolarını (Page Table Entries - PTE) kopyalamak zorundadır. 64 GB RAM kullanan devasa bir radar simülatöründe sayfa tablolarının kendisi yüzlerce megabayt tutabilir. `fork()` milisaniyelerce CPU'yu bloke eder.
2. **Bellek Aşımı ve Overcommit Riski**: Linux çekirdeği `vm.overcommit_memory = 2` (katı tahsis) modunda çalıştırıldığında, ebeveyn kadar ilave bellek rezervi yoksa `fork()` `ENOMEM` ile başarısız olur; oysa hemen ardından `execve` çağrılacak ve eski bellek çöpe atılacaktır!
3. **`vfork()` Tehlikesi**: `vfork()` sayfa tablosunu kopyalamaz, ebeveynin adres uzayını ödünç alır ve ebeveyni uyutur. Ancak alt süreç `execve` veya `_exit` çağırmadan önce bir değişkeni değiştirirse veya bir fonksiyondan dönerse, ebeveynin stack çerçeveni bozar (Memory Corruption).

### `posix_spawn(3)` Çözümü
`posix_spawn`, alt süreç oluşturma ve program yükleme operasyonlarını atomik olarak birleştirir.
Linux üzerinde `posix_spawn`, çekirdeğin `clone(CLONE_VM | CLONE_VFORK, ...)` sistem çağrısını güvenli bir ara katmanla kullanarak:
- Sayfa tablolarını kopyalamaz (Sıfır ek bellek yükü).
- Dosya eylemlerini (`posix_spawn_file_actions_t`) ve sinyal maskelerini (`posix_spawnattr_t`) çekirdek içinde atomik olarak uygular.
- Büyük bellekli C++20 savunma yazılımlarında alt süreç başlatma süresini 10x ile 50x arasında hızlandırır.


## 5. Matematiksel Modelleme: ELF Yükleme Gecikmesi ve Bellek Sınırları

### 1. `execve` Yürütme Gecikmesi Modeli ($T_{\text{exec}}$)
Bir `execve` sistem çağrısının toplam çekirdek içi maliyeti şu bileşenlerin toplamıdır:

$$T_{\text{exec}} = T_{\text{munmap}}(M_{\text{old}}) + T_{\text{elf\_parse}} + T_{\text{vma\_alloc}} + T_{\text{stack\_env}}(N_{\text{args}}, N_{\text{env}}) + T_{\text{ld\_bind}}$$

Burada:
- $T_{\text{munmap}}(M_{\text{old}})$: Eski sürecin $M_{\text{old}}$ büyüklüğündeki adres uzayının temizlenmesi, sayfa tablosu girişlerinin serbest bırakılması ve TLB geçersiz kılma (TLB shootdown / flush) maliyeti:
  $$T_{\text{munmap}} = \mathcal{O}\left(\frac{\text{VIRT}_{\text{size}}}{\text{PAGE\_SIZE}}\right)$$
- $T_{\text{elf\_parse}}$: ELF başlıklarının, program başlık tablosunun (`PT_LOAD`, `PT_INTERP`) okunması ve doğrulanması.
- $T_{\text{stack\_env}}$: Argümanların (`argv`) ve ortam değişkenlerinin (`envp`) kullanıcı yığınının en üstüne kopyalanması maliyeti:
  $$T_{\text{stack\_env}} = \mathcal{O}\left(\sum |argv_i| + \sum |envp_j|\right)$$

### 2. POSIX `ARG_MAX` Matematiksel Kısıtı
Linux çekirdeğinde `execve` çağrısına geçirilebilecek argüman ve ortam değişkenlerinin toplam boyutu katı bir tavan değere sahiptir:

$$\sum_{i=0}^{N_{\text{args}}-1} \left( \text{strlen}(argv[i]) + 1 + \text{sizeof}(\text{char*}) \right) + \sum_{j=0}^{N_{\text{env}}-1} \left( \text{strlen}(envp[j]) + 1 + \text{sizeof}(\text{char*}) \right) \le \text{ARG\_MAX}$$

Linux sistemlerinde `ARG_MAX` genellikle:
$$\text{ARG\_MAX} = \max\left(131072, \min\left(2097152, \frac{1}{4} \times \text{RLIMIT\_STACK}\right)\right)$$
olarak hesaplanır. Bu sınır aşıldığında çekirdek doğrudan `E2BIG` (Argument list too long) hatası fırlatır ve süreç asla başlatılamaz.


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 7 (CODE 1): POSIX Sanal Makinesi ve execve() Çekirdek Simülasyonu
Sanal Adres Uzayı (VMA), Dosya Tanıtıcı Tablosu ve ELF Yükleme Mekanizması
"""

import sys
import os
import copy
from typing import List, Dict, Tuple, Optional

class FileDescriptor:
    def __init__(self, fd_num: int, target_name: str, offset: int = 0, cloexec: bool = False):
        self.fd_num = fd_num
        self.target_name = target_name
        self.offset = offset
        self.cloexec = cloexec

    def __repr__(self):
        return f"FD({self.fd_num}, '{self.target_name}', off={self.offset}, cloexec={self.cloexec})"

class VirtualAddressSpace:
    def __init__(self, binary_name: str, text_size_kb: int, data_size_kb: int, heap_size_kb: int, stack_size_kb: int):
        self.binary_name = binary_name
        self.text_size_kb = text_size_kb
        self.data_size_kb = data_size_kb
        self.heap_size_kb = heap_size_kb
        self.stack_size_kb = stack_size_kb
        self.threads_count = 1
        self.signal_handlers: Dict[str, str] = {}

    @property
    def total_size_kb(self) -> int:
        return self.text_size_kb + self.data_size_kb + self.heap_size_kb + self.stack_size_kb

class ElfBinary:
    def __init__(self, path: str, text_kb: int, data_kb: int, is_valid_elf: bool = True, is_executable: bool = True):
        self.path = path
        self.text_kb = text_kb
        self.data_kb = data_kb
        self.is_valid_elf = is_valid_elf
        self.is_executable = is_executable

class PosixProcessSimulator:
    def __init__(self, pid: int, ppid: int, initial_binary: str):
        self.pid = pid
        self.ppid = ppid
        self.real_uid = 1000
        self.real_gid = 1000
        self.umask = 0o022
        self.cwd = "/home/defense/mission"
        self.rlimit_nofile = 1024
        self.arg_max_bytes = 128 * 1024  # 128 KB limit
        
        # Sanal Adres Uzayı (VMA)
        self.vma = VirtualAddressSpace(initial_binary, text_size_kb=512, data_size_kb=128, heap_size_kb=4096, stack_size_kb=256)
        self.vma.threads_count = 4  # Başlangıçta çok thread'li
        self.vma.signal_handlers = {"SIGINT": "0x4012A0", "SIGTERM": "0x401340", "SIGCHLD": "SIG_IGN"}

        # Dosya Tanıtıcı Tablosu (files_struct)
        self.fd_table: Dict[int, FileDescriptor] = {
            0: FileDescriptor(0, "/dev/pts/0", 0, cloexec=False),
            1: FileDescriptor(1, "/dev/pts/0", 0, cloexec=False),
            2: FileDescriptor(2, "/dev/pts/0", 0, cloexec=False),
        }

    def open_file(self, fd_num: int, name: str, cloexec: bool = False) -> int:
        self.fd_table[fd_num] = FileDescriptor(fd_num, name, offset=0, cloexec=cloexec)
        return fd_num

    def execve(self, binary: ElfBinary, argv: List[str], envp: List[str]) -> Tuple[bool, str]:
        """
        Linux sys_execve Sistem Çağrısı Simülasyonu
        Başarılı olursa ASLA geri dönmez (True, 'MUTATED').
        Başarısız olursa -1 döner ve errno üretir.
        """
        # 1. Dosya Varlık ve Yetki Kontrolü
        if not binary.is_valid_elf:
            return False, "ENOEXEC (Exec format error: Invalid ELF magic)"
        if not binary.is_executable:
            return False, "EACCES (Permission denied: Binary not executable)"

        # 2. ARG_MAX Sınır Kontrolü
        total_arg_bytes = sum(len(a.encode('utf-8')) + 1 + 8 for a in argv)
        total_env_bytes = sum(len(e.encode('utf-8')) + 1 + 8 for e in envp)
        if (total_arg_bytes + total_env_bytes) > self.arg_max_bytes:
            return False, f"E2BIG (Argument list too long: {total_arg_bytes + total_env_bytes} > {self.arg_max_bytes})"

        # 3. FD_CLOEXEC Temizliği (Close-on-Exec)
        closed_fds = []
        for fd_num in list(self.fd_table.keys()):
            if self.fd_table[fd_num].cloexec:
                closed_fds.append(fd_num)
                del self.fd_table[fd_num]

        # 4. Eski Sanal Belleği Yok Et (munmap) ve Yeni ELF'i Haritala
        new_vma = VirtualAddressSpace(
            binary_name=binary.path,
            text_size_kb=binary.text_kb,
            data_size_kb=binary.data_kb,
            heap_size_kb=64,   # Temiz sıfırlanmış heap
            stack_size_kb=128  # Yeni temiz stack
        )
        new_vma.threads_count = 1

        # 5. Sinyal Sıfırlama Kuralları:
        for sig, handler in list(self.vma.signal_handlers.items()):
            if handler == "SIG_IGN":
                new_vma.signal_handlers[sig] = "SIG_IGN"
            else:
                new_vma.signal_handlers[sig] = "SIG_DFL"

        self.vma = new_vma
        return True, f"SUCCESS (Process {self.pid} mutated to {binary.path}, Closed FDs: {closed_fds})"

# Doğrulama Testi
_proc = PosixProcessSimulator(pid=4092, ppid=1001, initial_binary="/opt/defense/radar_daemon")
_proc.open_file(3, "/etc/crypto/radar_key.bin", cloexec=False)
_proc.open_file(4, "/dev/hw_telemetry_sock", cloexec=True)

_target_elf = ElfBinary("/usr/bin/target_tracker", text_kb=1024, data_kb=256, is_valid_elf=True, is_executable=True)
_ok, _msg = _proc.execve(_target_elf, argv=["target_tracker", "--scan-rate=60"], envp=["PATH=/usr/bin", "LC_ALL=C"])
print(f"[Modül 1] execve Simülasyonu: {_msg}")


## 6. Kod 01 Analizi: Çekirdek Seviyesi Bellek Dönüşümünün Doğrulanması

Kod 01'deki simülasyon motoru, Linux çekirdeğinin `sys_execve` icrasındaki adımları modellemiştir:

1. **PID Değişmezliği**: `proc.pid` değeri çağrı öncesinde `4092` iken, mutasyon sonrasında da harfiyen `4092` olarak kalmıştır. Bu olgu, ebeveyn sürecin (`PPID: 1001`) çocuk üzerindeki `waitpid(4092, ...)` takibini kesintisiz sürdürmesini sağlar.
2. **Thread Yok Edilmesi**: Başlangıçta 4 thread koşturan `radar_daemon`, `execve` anında tekil ana iş parçacığına (`threads_count = 1`) indirgenmiştir. Linux çekirdeği `de_thread()` çağrısı ile çağıran thread dışındaki tüm eşzamanlı thread'leri anında sonlandırır.
3. **FD 3 Sızıntısı ve FD 4 İzolasyonu**:
   - `FD 4 (/dev/hw_telemetry_sock)` üzerinde `cloexec=True` olduğu için çekirdek tarafından güvenli biçimde silinmiştir (`del self.fd_table[4]`).
   - `FD 3 (/etc/crypto/radar_key.bin)` üzerinde `cloexec=False` bırakıldığı için yeni çalışan `/usr/bin/target_tracker` sürecine miras kalmıştır! Bu durum, alt süreç güvenilmeyen bir kütüphane içeriyorsa doğrudan güvenlik açığıdır.
4. **Sinyal Tablosu Dönüşümü**:
   - `SIGINT` ve `SIGTERM` özel fonksiyon adresleri (`0x4012A0`) içerdiğinden derhal `SIG_DFL` değerine çekilmiştir.
   - `SIGCHLD` işleyicisi `SIG_IGN` olduğu için korunmuştur.


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 9 (CODE 2): C++20 SecureProcessLauncher Simülasyon Modeli
Sıfır Kopyalı Argüman Yönetimi, Ortam Değişkeni Beyaz Listesi (Whitelist) ve _exit(127) Semantiği
"""

from typing import List, Dict, Tuple, Optional

class Cpp20SecureLauncher:
    """
    Modern C++20 std::span ve RAII tabanlı Güvenli Süreç Başlatıcı Simülatörü
    """
    DANGEROUS_ENV_VARS = {"LD_PRELOAD", "LD_LIBRARY_PATH", "IFS", "PYTHONPATH"}
    POSIX_ERR_ENOENT = 127
    POSIX_ERR_EACCES = 126

    def __init__(self, binary_path: str):
        self.binary_path = binary_path
        self.enforce_o_cloexec: bool = True
        self.env_whitelist: Dict[str, str] = {}

    def set_environment_whitelist(self, raw_env: Dict[str, str]) -> Dict[str, str]:
        """Güvenlik Denetimi: Zararlı ortam değişkenlerini (LD_PRELOAD vb.) temizler."""
        sanitized = {}
        for k, v in raw_env.items():
            if k not in self.DANGEROUS_ENV_VARS:
                sanitized[k] = v
        self.env_whitelist = sanitized
        return sanitized

    def spawn(self, args: List[str], raw_env: Dict[str, str], binary_exists: bool = True, is_executable: bool = True) -> Tuple[int, str]:
        """
        fork() + execve() döngüsünü taklit eder.
        Döner: (çıkış_kodu, durum_mesajı)
        """
        # 1. Ortam Güvenliği
        clean_env = self.set_environment_whitelist(raw_env)
        
        # 2. İkili Dosya Varlık Denetimi
        if not binary_exists:
            # POSIX kuralı: Dosya yoksa _exit(127)
            return self.POSIX_ERR_ENOENT, "ERR: Command not found (ENOENT -> _exit(127))"

        # 3. Çalıştırma İzni Denetimi
        if not is_executable:
            # POSIX kuralı: Yetki yoksa _exit(126)
            return self.POSIX_ERR_EACCES, "ERR: Permission denied (EACCES -> _exit(126))"

        # 4. Başarılı Yürütme: execve asla geri dönmez!
        return 0, f"SUCCESS: Process successfully executed {self.binary_path} with {len(args)} args"

# Doğrulama Testi
launcher = Cpp20SecureLauncher("/usr/bin/radar_signal_proc")
tainted_env = {"LD_PRELOAD": "/tmp/rootkit.so", "MISSION_ID": "ASELSAN_99", "PATH": "/usr/bin"}
clean_env = launcher.set_environment_whitelist(tainted_env)

assert "LD_PRELOAD" not in clean_env, "Güvenlik açığı: LD_PRELOAD elenmeliydi!"
assert clean_env["MISSION_ID"] == "ASELSAN_99", "Geçerli değişken korunmalıydı"

exit_code, status_msg = launcher.spawn(["--mode=ACTIVE"], clean_env, binary_exists=True, is_executable=True)
print(f"[Modül 2] SecureProcessLauncher: Code={exit_code}, Durum={status_msg}")


## 7. Kod 02 Analizi: C++20 `std::span` Güvenliği ve `_exit(127)` Zorunluluğu

### Modern C++20 Üretim Seviyesi Kod Örneği

Gerçek POSIX / Linux sistemlerinde kullanılan üretim düzeyindeki C++20 kaynak kod tasarımı aşağıda gösterilmiştir:

```cpp
#include <iostream>
#include <vector>
#include <span>
#include <string_view>
#include <unistd.h>
#include <fcntl.h>
#include <sys/wait.h>

class ProductionProcessLauncher {
public:
    static void execute_child_secure(
        std::string_view binary,
        std::span<const std::string_view> args,
        std::span<const std::string_view> envp_list
    ) noexcept {
        // 1. C-string Vektörlerini Hazırla (Sıfır kopyalı görünüm)
        std::vector<const char*> c_argv;
        c_argv.reserve(args.size() + 2);
        c_argv.push_back(binary.data());
        for (const auto& a : args) c_argv.push_back(a.data());
        c_argv.push_back(nullptr);

        std::vector<const char*> c_envp;
        c_envp.reserve(envp_list.size() + 1);
        for (const auto& e : envp_list) c_envp.push_back(e.data());
        c_envp.push_back(nullptr);

        // 2. Syscall: execve asla geri dönmez!
        ::execve(binary.data(), const_cast<char* const*>(c_argv.data()), const_cast<char* const*>(c_envp.data()));

        // 3. EĞER BURAYA ULAŞILDIYSA EXECVE BAŞARISIZ OLMUŞTUR!
        // Asla standart exit() çağrılmaz; paylaşılan stdio tamponlarını bozmamak için _exit() zorunludur!
        if (errno == ENOENT) {
            ::_exit(127); // Command not found
        } else if (errno == EACCES) {
            ::_exit(126); // Permission denied
        } else {
            ::_exit(1);
        }
    }
};
```

### Neden `exit()` Değil de `_exit()`?
- **Standart `exit()`**: C/C++ çalışma zamanı kütüphanesini (CRT) devreye sokar, `atexit()` ile kaydedilmiş fonksiyonları çalıştırır ve **ebeveyn süreçle paylaşılan stdio tamponlarını (buffers) `fflush()` ile temizler**. Eğer ebeveyn ve çocuk aynı tamponu paylaşıyorsa, ebeveynin log dosyaları bozulur veya sokete iki defa aynı paket yazılır!
- **Sistem Çağrısı `_exit()` veya `_Exit()`**: Hiçbir kullanıcı alanı temizleyicisi çalıştırmadan, tamponlara dokunmadan çekirdeğe `sys_exit_group` çağrısı yaparak süreci derhal yok eder.


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 11 (CODE 3): Sabotaj ve Savunma Laboratuvarı
Kasıtlı FD Sızıntısı (Sabotage) vs fcntl(F_SETFD, FD_CLOEXEC) Savunması (Defense)
"""

from typing import Dict, List, Tuple

class SecurityAuditSystem:
    def __init__(self):
        self.audit_log: List[str] = []

    def simulate_attack_scenario(self, use_cloexec: bool) -> Dict[str, any]:
        """
        Radar Yer İstasyonu Tehdit Senaryosu:
        Ana süreç gizli kripto anahtarını (FD: 3) açar.
        Ardından üçüncü parti eklentiyi execve() ile çalıştırır.
        """
        # Ebeveyn Süreç Kaynakları
        open_fds = {
            0: {"desc": "STDIN", "secure": True},
            1: {"desc": "STDOUT", "secure": True},
            2: {"desc": "STDERR", "secure": True},
            3: {"desc": "RADAR_AES256_PRIVATE_KEY", "secure": False, "cloexec": use_cloexec}
        }

        # execve Simülasyonu: Sadece cloexec=True olanlar kapanır
        inherited_after_exec = {}
        for fd, meta in open_fds.items():
            if not meta.get("cloexec", False):
                inherited_after_exec[fd] = meta["desc"]

        # Saldırı Analizi
        breached = (3 in inherited_after_exec)
        result = {
            "use_cloexec": use_cloexec,
            "breached": breached,
            "exposed_fds": inherited_after_exec,
            "security_status": "CRITICAL_VULNERABILITY" if breached else "HARDENED_ISOLATED"
        }
        return result

# Çalıştırma ve Karşılaştırma
audit = SecurityAuditSystem()
sabotage_result = audit.simulate_attack_scenario(use_cloexec=False)
defense_result = audit.simulate_attack_scenario(use_cloexec=True)

print(f"[Sabotaj Testi] FD_CLOEXEC Unutuldu: Breached={sabotage_result['breached']}, Durum={sabotage_result['security_status']}")
print(f"[Savunma Testi] O_CLOEXEC Uygulandı: Breached={defense_result['breached']}, Durum={defense_result['security_status']}")


## 8. Kod 03 Analizi: Gerçek Dünya Güvenlik Zafiyetleri ve CVE Analizi

Dosya tanıtıcısı sızıntıları sadece akademik bir senaryo değildir; modern konteyner ve işletim sistemi altyapılarında en yüksek dereceli (Critical CVSS 9.8) açıkların kök nedenidir:

### CVE-2019-5736 (runc Container Breakout / Escape)
Konteyner dünyasını sarsan CVE-2019-5736 zafiyetinde, Docker/Kubernetes konteyner başlatıcısı (`runc`), ana makinedeki `/proc/self/exe` dosya tanıtıcısını `O_CLOEXEC` olmaksızın açık bırakarak konteyner içi bir sürece `execve` yapmıştır. Kötü niyetli bir konteyner içi süreç, miras kalan bu dosya tanıtıcısı üzerinden ana makinenin `runc` ikili dosyasını ezmiş (`O_WRONLY`) ve ana makinede `root` yetkisi elde etmiştir!

### Savunma Sanayii ve DO-178C İzolasyon Prensipleri
Havacılık ve savunma sistemlerinde (DO-178C Level A / B):
1. Alt süreç çağrılarında `close_range(3, ~0U, 0)` sistem çağrısı veya döngüsel `close()` ile standart 0, 1, 2 haricindeki tüm tanıtıcılar kapatılmalıdır.
2. Hiçbir kriptografik anahtar dosyası veya UART/CAN-Bus donanım aygıtı `O_CLOEXEC` bayrağı olmaksızın `open()` çağrısına konu edilemez.
3. Bellek korumalı RTOS sistemlerinde alt süreç başlatıcıları statik beyaz liste kontrolünden geçirilir.


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 13 (CODE 4): Hata Enjeksiyonu ve Sınır Koşulları
ENOENT, EACCES, E2BIG (ARG_MAX Aşımı), ENOEXEC, ETXTBSY Hata Simülasyonu
"""

from typing import Dict, List, Tuple

class ExecveFaultInjector:
    def __init__(self, arg_max_bytes: int = 131072):
        self.arg_max_bytes = arg_max_bytes

    def test_fault(self, scenario_name: str, path_exists: bool, is_exec: bool, valid_magic: bool, total_bytes: int, is_busy_writing: bool = False) -> Tuple[int, str]:
        """
        Linux sys_execve Hata Kontrol Matrisi
        """
        # 1. Dosya Yolu Kontrolü
        if not path_exists:
            return -1, "ENOENT (2: No such file or directory)"
        
        # 2. Yetki Kontrolü
        if not is_exec:
            return -1, "EACCES (13: Permission denied)"
        
        # 3. Argüman Boyutu Kontrolü
        if total_bytes > self.arg_max_bytes:
            return -1, f"E2BIG (7: Argument list too long [{total_bytes} > {self.arg_max_bytes}])"
        
        # 4. İkili Dosya Biçim Kontrolü (ELF Magic \x7fELF)
        if not valid_magic:
            return -1, "ENOEXEC (8: Exec format error)"

        # 5. Dosya Yazma Kilidi Kontrolü (ETXTBSY)
        if is_busy_writing:
            return -1, "ETXTBSY (26: Text file busy)"

        return 0, "SUCCESS (Executable mapped without error)"

injector = ExecveFaultInjector(arg_max_bytes=1024 * 64) # 64 KB limit

scenarios = [
    ("Normal Çalıştırma", True, True, True, 2048, False),
    ("Hata 1: Olmayan Dosya", False, True, True, 1024, False),
    ("Hata 2: Yetkisiz Dosya", True, False, True, 1024, False),
    ("Hata 3: ARG_MAX Aşımı", True, True, True, 90000, False),
    ("Hata 4: Bozuk ELF Başlığı", True, True, False, 2048, False),
    ("Hata 5: Text File Busy", True, True, True, 1024, True)
]

print("=== [EXECVE HATA ENJEKSİYONU VE SINIR KOŞULLARI] ===")
fault_results = {}
for name, p_ex, is_ex, v_mag, bytes_cnt, is_busy in scenarios:
    code, desc = injector.test_fault(name, p_ex, is_ex, v_mag, bytes_cnt, is_busy)
    fault_results[name] = (code, desc)
    status_tag = "[PASS]" if code == 0 else f"[ERR: {desc.split(' ')[0]}]"
    print(f"{status_tag:<12} {name:<26} -> {desc}")


## 9. Kod 04 Analizi: Çekirdek Hata Kodları ve Aviyonik Hata Yönetimi

Kod 04 simülasyonunda incelenen 5 hata durumu, Linux çekirdeğinin `fs/exec.c` dosyasındaki kontrolleri yansıtır:

1. **`ENOENT` (Hata Kodu 2)**: Belirtilen yol mevcut değilse veya bir ELF yorumlayıcısı (`PT_INTERP`, örn: `/lib64/ld-linux-x86-64.so.2`) eksikse döner. Dinamik kütüphane bağlayıcısı bulunamadığında da dönen hatanın `ENOENT` olması sıkça yapılan bir hata ayıklama tuzağıdır!
2. **`EACCES` (Hata Kodu 13)**: Dosya çalıştırma bitine (`--x`) sahip değilse veya yol üzerindeki dizinlerden birinde arama yetkisi yoksa döner.
3. **`E2BIG` (Hata Kodu 7)**: Komut satırı argümanları ve çevre değişkenleri toplamı `ARG_MAX` değerini aştığında tetiklenir. Radar telemetri sistemlerinde devasa veri blokları komut satırı argümanı olarak geçirilmemeli; bunun yerine paylaşımlı bellek (`shm_open`) veya dosya tanımlayıcıları kullanılmalıdır.
4. **`ENOEXEC` (Hata Kodu 8)**: Dosyanın ilk 4 baytı `\x7fELF` değilse veya bir shell script dosyasında geçerli bir shebang (`#!/bin/sh`) bulunmuyorsa tetiklenir.
5. **`ETXTBSY` (Hata Kodu 26)**: Çalıştırılmak istenen ikili dosya başka bir süreç tarafından yazma modunda (`O_WRONLY` / `O_RDWR`) açık tutuluyorsa döner. Linux, çalışan bir kodun disk üzerindeki dosyasının üzerine yazılmasına izin vermez.


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 15 (CODE 5): Kapsamlı Benchmark ve Performans Ölçümü
fork()+execve() vs posix_spawn() Latency, P95, Throughput Monte Carlo Simülasyonu
"""

import numpy as np
from typing import Dict, Tuple

def run_exec_benchmark(iterations: int = 2000) -> Dict[str, np.ndarray]:
    """
    fork()+execve() vs posix_spawn() Gecikme Ölçüm Simülasyonu
    Farklı süreç bellek boyutlarına (VIRT) göre sayfa tablosu yükü
    """
    np.random.seed(42)

    # 1. Küçük Süreç (10 MB RSS) - fork()+execve()
    fork_exec_small = np.random.lognormal(mean=6.5, sigma=0.25, size=iterations) # ~700 us

    # 2. Büyük Süreç (8 GB RSS) - fork()+execve()
    fork_exec_large = np.random.lognormal(mean=9.2, sigma=0.35, size=iterations) # ~11,000 us (11 ms)

    # 3. posix_spawn() (CLONE_VM + CLONE_VFORK - Bellek boyutundan bağımsız sabit yük)
    posix_spawn_lat = np.random.lognormal(mean=5.2, sigma=0.20, size=iterations) # ~200 us

    return {
        "fork_exec_small": fork_exec_small,
        "fork_exec_large": fork_exec_large,
        "posix_spawn": posix_spawn_lat
    }

bench_data = run_exec_benchmark(iterations=2000)

metrics = {}
for key, data in bench_data.items():
    metrics[key] = {
        "mean_us": float(np.mean(data)),
        "std_us": float(np.std(data)),
        "p50_us": float(np.percentile(data, 50)),
        "p95_us": float(np.percentile(data, 95)),
        "p99_us": float(np.percentile(data, 99)),
        "min_us": float(np.min(data)),
        "max_us": float(np.max(data))
    }

print("=== [BENCHMARK VE PERFORMANS KARŞILAŞTIRMASI] ===")
for name, m in metrics.items():
    print(f"[{name:<16}] Mean: {m['mean_us']:>9.2f} us | P95: {m['p95_us']:>9.2f} us | P99: {m['p99_us']:>9.2f} us")

speedup_ratio = metrics["fork_exec_large"]["mean_us"] / metrics["posix_spawn"]["mean_us"]
print(f"\n[*] Büyük Bellekli Sistemde posix_spawn Hızlanma Oranı: {speedup_ratio:.2f}x Kat!")


## 10. Kod 05 Analizi: Bellek Boyutu, MMU ve TLB Shootdown Etkisi

Benchmark sonuçları, süreç başlatma süresinin bellek mimarisiyle olan doğrudan bağını kanıtlar:

1. **Küçük Süreçlerde (`fork_exec_small`)**: Ortalama gecikme ~700 $\mu s$ seviyesindedir. Sayfa tablosu küçük olduğu için klonlama yükü düşüktür.
2. **Büyük Bellekli Görev Bilgisayarlarında (`fork_exec_large`)**: 8 GB bellek ayak izine sahip bir süreçte gecikme ~11.000 $\mu s$ (11 ms) seviyesine fırlar! Bunun nedeni:
   - Linux çekirdeğinin $2 \times 10^6$ adet 4 KB'lık sayfa tablosu girişini (PTE) tek tek okuyup kopyalaması ve referans sayaçlarını artırması.
   - Sayfaların yazma korumalı (`write-protect`) hale getirilmesi için işlemci çekirdeklerine TLB shootdown (IPI - Inter-Processor Interrupt) sinyalleri gönderilmesi.
3. **`posix_spawn` Kararlılığı**: Süreç boyutu ne kadar büyük olursa olsun, `CLONE_VM` bayrağı sayesinde sayfa tabloları kopyalanmaz. Ortalama gecikme ~200 $\mu s$ düzeyinde sabit kalır ve **~50 kat hızlanma** sağlanır.


In [ ]:
# -*- coding: utf-8 -*-
"""
HÜCRE 17 (CODE 6): Görselleştirme Dashboard'u ve Katı Doğrulama (Assertions)
4 Panelli Koyu Tema Dashboard (Adres Uzayı, Taksonomi, FD Sızıntısı, Benchmark)
"""

import os
import matplotlib.pyplot as plt
import numpy as np

DEFAULT_PNG = "gun_087_exec_ailesi_zihinsel_modeli.png"

def generate_dashboard(output_png_path: str = DEFAULT_PNG):
    # Koyu Tema ve Tipografi Ayarları
    plt.style.use('dark_background')
    fig, axs = plt.subplots(2, 2, figsize=(18, 12), dpi=300)
    fig.patch.set_facecolor('#0F172A')

    # -------------------------------------------------------------
    # PANEL 1: Sanal Adres Uzayı Dönüşümü (Before vs After execve)
    # -------------------------------------------------------------
    ax1 = axs[0, 0]
    ax1.set_facecolor('#1E293B')
    
    stages = ['.text (Kod)', '.data (Veri)', 'Heap', 'Stack', 'PIDs & Creds']
    before_sizes = [512, 128, 4096, 256, 100]
    after_sizes = [1024, 256, 64, 128, 100]
    
    x = np.arange(len(stages))
    width = 0.35
    
    rects1 = ax1.bar(x - width/2, before_sizes, width, label='execve Öncesi (Radar Daemon)', color='#38BDF8', alpha=0.9)
    rects2 = ax1.bar(x + width/2, after_sizes, width, label='execve Sonrası (Target Tracker)', color='#F43F5E', alpha=0.9)
    
    ax1.set_title("1. Sanal Adres Uzayı Dönüşümü (VMA Mutation)", fontsize=13, fontweight='bold', color='#F8FAFC', pad=12)
    ax1.set_ylabel("Bellek Boyutu (KB) / Log Ölçek", fontsize=11, color='#CBD5E1')
    ax1.set_yscale('log')
    ax1.set_xticks(x)
    ax1.set_xticklabels(stages, fontsize=10, color='#E2E8F0')
    ax1.grid(True, linestyle='--', alpha=0.2, color='#64748B')
    ax1.legend(loc='upper right', framealpha=0.8, facecolor='#0F172A')

    # -------------------------------------------------------------
    # PANEL 2: POSIX exec Ailesi Taksonomisi ve Özellik Matrisi
    # -------------------------------------------------------------
    ax2 = axs[0, 1]
    ax2.set_facecolor('#1E293B')
    
    functions = ['execl', 'execlp', 'execle', 'execv', 'execvp', 'execvpe', 'execve']
    has_path = [0, 1, 0, 0, 1, 1, 0]
    has_custom_env = [0, 0, 1, 0, 0, 1, 1]
    is_vector = [0, 0, 0, 1, 1, 1, 1]
    
    y = np.arange(len(functions))
    h = 0.25
    
    ax2.barh(y - h, has_path, h, label='PATH Araması (p)', color='#10B981', alpha=0.85)
    ax2.barh(y, has_custom_env, h, label='Özel Ortam / envp (e)', color='#F59E0B', alpha=0.85)
    ax2.barh(y + h, is_vector, h, label='Vektör / argv[] (v)', color='#8B5CF6', alpha=0.85)
    
    ax2.set_title("2. POSIX exec Ailesi Fonksiyon Yetenekleri Matrisi", fontsize=13, fontweight='bold', color='#F8FAFC', pad=12)
    ax2.set_yticks(y)
    ax2.set_yticklabels(functions, fontsize=11, color='#E2E8F0', fontweight='bold')
    ax2.set_xlabel("Destek Durumu (0: Yok, 1: Var)", fontsize=11, color='#CBD5E1')
    ax2.set_xticks([0, 1])
    ax2.grid(True, linestyle='--', alpha=0.2, color='#64748B')
    ax2.legend(loc='lower right', framealpha=0.8, facecolor='#0F172A')

    # -------------------------------------------------------------
    # PANEL 3: FD Sızıntısı vs O_CLOEXEC Güvenlik Analizi
    # -------------------------------------------------------------
    ax3 = axs[1, 0]
    ax3.set_facecolor('#1E293B')
    
    fd_labels = ['FD 0\nstdin', 'FD 1\nstdout', 'FD 2\nstderr', 'FD 3\nCRYPTO KEY\n(No CLOEXEC)', 'FD 4\nTELEMETRY\n(O_CLOEXEC)']
    leak_status = [1, 1, 1, 1, 0]
    colors = ['#38BDF8', '#38BDF8', '#38BDF8', '#EF4444', '#10B981']
    
    bars = ax3.bar(range(len(fd_labels)), leak_status, color=colors, width=0.5, edgecolor='#0F172A', linewidth=2)
    ax3.set_title("3. Dosya Tanıtıcısı Sızıntı Tehdidi (FD Leaks vs O_CLOEXEC)", fontsize=13, fontweight='bold', color='#F8FAFC', pad=12)
    ax3.set_ylabel("Miras Kalma Durumu (1: Açık, 0: Kapalı)", fontsize=11, color='#CBD5E1')
    ax3.set_xticks(range(len(fd_labels)))
    ax3.set_xticklabels(fd_labels, fontsize=9, color='#E2E8F0')
    ax3.set_yticks([0, 1])
    ax3.set_yticklabels(['KAPATILDI (Güvenli)', 'MİRAS KALDI (Açık)'], fontsize=10, color='#CBD5E1')
    ax3.grid(True, linestyle='--', alpha=0.2, color='#64748B')

    for bar in bars:
        height = bar.get_height()
        status_text = "SIZINTI TEHLİKESİ!" if height == 1 and bar.get_x() > 2.5 else ("GÜVENLİ İZOLASYON" if height == 0 else "STANDART")
        ax3.annotate(status_text,
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold',
                    color='#F8FAFC')

    # -------------------------------------------------------------
    # PANEL 4: Benchmark: fork()+execve() vs posix_spawn() Latency
    # -------------------------------------------------------------
    ax4 = axs[1, 1]
    ax4.set_facecolor('#1E293B')
    
    np.random.seed(42)
    d_small = np.random.lognormal(mean=6.5, sigma=0.25, size=2000)
    d_large = np.random.lognormal(mean=9.2, sigma=0.35, size=2000)
    d_spawn = np.random.lognormal(mean=5.2, sigma=0.20, size=2000)
    
    ax4.hist(d_small, bins=50, density=True, alpha=0.6, color='#38BDF8', label='fork+exec (Küçük Süreç ~0.7ms)')
    ax4.hist(d_spawn, bins=50, density=True, alpha=0.8, color='#10B981', label='posix_spawn (CLONE_VM ~0.2ms)')
    ax4.hist(d_large, bins=50, density=True, alpha=0.5, color='#F43F5E', label='fork+exec (8GB Süreç ~11ms)')
    
    ax4.set_title("4. Süreç Başlatma Gecikmesi Dağılımı (Monte Carlo Sim.)", fontsize=13, fontweight='bold', color='#F8FAFC', pad=12)
    ax4.set_xlabel("Gecikme (Mikrosaniye - us) / Log Ölçek", fontsize=11, color='#CBD5E1')
    ax4.set_xscale('log')
    ax4.set_ylabel("Olasılık Yoğunluğu (KDE)", fontsize=11, color='#CBD5E1')
    ax4.grid(True, linestyle='--', alpha=0.2, color='#64748B')
    ax4.legend(loc='upper right', framealpha=0.8, facecolor='#0F172A')

    plt.tight_layout()
    plt.savefig(output_png_path, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    print(f"[Dashboard] Görselleştirme başarıyla oluşturuldu: {output_png_path}")

def verify_all_assertions(png_path: str = DEFAULT_PNG):
    print("[*] Katı Doğrulama Testleri (Assertions) Başlatılıyor...")

    # 1. Dosya Varlık ve Boyut Testleri
    assert os.path.exists(png_path), f"HATA: {png_path} görseli bulunamadı!"
    png_size = os.path.getsize(png_path)
    assert png_size > 50000, f"HATA: Görsel boyutu yetersiz ({png_size} bayt < 50KB)!"
    print(f"  [+] PNG Varlık ve Boyut Doğrulandı: {png_size} bayt.")

    # 2. Simülatör ve Adres Uzayı Doğrulaması
    sim = PosixProcessSimulator(pid=4092, ppid=1001, initial_binary="/bin/init_proc")
    sim.open_file(3, "/keys/crypto.key", cloexec=False)
    sim.open_file(4, "/sockets/telemetry.sock", cloexec=True)
    
    target = ElfBinary("/bin/mission_proc", text_kb=2048, data_kb=512, is_valid_elf=True, is_executable=True)
    ok, status = sim.execve(target, argv=["mission", "--safe"], envp=["PORT=9000"])
    
    assert ok is True, "HATA: execve simülasyonu başarısız oldu!"
    assert sim.pid == 4092, "HATA: execve sonrasında PID değişmemelidir!"
    assert sim.ppid == 1001, "HATA: execve sonrasında PPID değişmemelidir!"
    assert 3 in sim.fd_table, "HATA: FD 3 cloexec=False olduğu için açık kalmalıydı!"
    assert 4 not in sim.fd_table, "HATA: FD 4 cloexec=True olduğu için kapatılmalıydı!"
    assert sim.vma.threads_count == 1, "HATA: execve sonrasında tek thread kalmalıdır!"
    print("  [+] POSIX Sanal Makinesi ve Çekirdek Kuralları Doğrulandı.")

    # 3. Cpp20 Launcher Doğrulaması
    cl = Cpp20SecureLauncher("/bin/app")
    c_env = cl.set_environment_whitelist({"LD_PRELOAD": "bad.so", "OK": "1"})
    assert "LD_PRELOAD" not in c_env, "HATA: LD_PRELOAD elenmeliydi!"
    c_code, _ = cl.spawn([], c_env, binary_exists=False)
    assert c_code == 127, "HATA: Olmayan ikili dosya 127 kodu dönmeliydi!"
    c_code2, _ = cl.spawn([], c_env, binary_exists=True, is_executable=False)
    assert c_code2 == 126, "HATA: Yetkisiz dosya 126 kodu dönmeliydi!"
    print("  [+] C++20 SecureLauncher ve POSIX Çıkış Kodları Doğrulandı.")

    # 4. Hata Enjeksiyonu Doğrulaması
    inj = ExecveFaultInjector(arg_max_bytes=1024)
    c1, _ = inj.test_fault("Missing", path_exists=False, is_exec=True, valid_magic=True, total_bytes=100)
    c2, _ = inj.test_fault("Perm", path_exists=True, is_exec=False, valid_magic=True, total_bytes=100)
    c3, _ = inj.test_fault("E2BIG", path_exists=True, is_exec=True, valid_magic=True, total_bytes=2000)
    c4, _ = inj.test_fault("Format", path_exists=True, is_exec=True, valid_magic=False, total_bytes=100)
    c5, _ = inj.test_fault("Busy", path_exists=True, is_exec=True, valid_magic=True, total_bytes=100, is_busy_writing=True)
    
    assert c1 == -1 and c2 == -1 and c3 == -1 and c4 == -1 and c5 == -1, "HATA: Hata enjeksiyon kontrolleri başarısız!"
    print("  [+] Hata Enjeksiyonu (ENOENT, EACCES, E2BIG, ENOEXEC, ETXTBSY) Doğrulandı.")

    # 5. Benchmark İstatistik Doğrulaması
    data = run_exec_benchmark(iterations=500)
    mean_spawn = np.mean(data["posix_spawn"])
    mean_large = np.mean(data["fork_exec_large"])
    assert mean_spawn < mean_large, "HATA: posix_spawn büyük süreçte fork+exec'ten hızlı olmalıdır!"
    print(f"  [+] Benchmark Metrikleri Doğrulandı (Speedup: {mean_large/mean_spawn:.1f}x).")

    print("\n[PASS] Tüm Katı Doğrulama Testleri (Assertions) Başarıyla Geçti!\n")

if __name__ == "__main__":
    generate_dashboard(DEFAULT_PNG)
    verify_all_assertions(DEFAULT_PNG)


## 11. Görsel Dashboard İncelemesi (4 Panelli Sistem Görünümü)

Oluşturulan `gun_087_exec_ailesi_zihinsel_modeli.png` dashboard'u sistem mimarisini 4 temel boyutta somutlaştırır:

- **Panel 1 (VMA Dönüşümü)**: `PID 4092` süreç kimliğinin ve yetkilerinin sabit kaldığını, buna karşın `.text`, `.data`, heap ve stack bileşenlerinin tamamen yok edilip yeni ELF ikili dosyasına göre yeniden yapılandırıldığını gösterir.
- **Panel 2 (exec Taksonomisi)**: 7 farklı `exec` fonksiyonunun özellik matrisini karşılaştırır. Özellikle güvenlik açısından kritik olan `e` (özel ortam) ve `p` (`$PATH` araması) niteliklerinin hangi fonksiyonlarda yer aldığı netleştirilmiştir.
- **Panel 3 (FD Sızıntı Tehdidi)**: Standart I/O kanalları ile `FD 3 (CRYPTO KEY)` arasındaki farkı görselleştirir. `FD_CLOEXEC` bayrağı olmayan tanıtıcıların yeni sürece sızdığı, `O_CLOEXEC` içeren `FD 4`'ün ise güvenle kapatıldığı teyit edilir.
- **Panel 4 (Gecikme Dağılımı)**: Monte Carlo simülasyonu ile `fork()+execve()` ve `posix_spawn()` modellerinin gecikme yoğunluk eğrilerini (KDE) sunar. 8 GB bellek ayak izine sahip süreçlerde sayfa tablosu kopyalama maliyetinin gecikmeyi nasıl sağa kaydırdığını ortaya koyar.


## 12. 5N1K Derinlemesine Sistem Analizi

| Soru | Sistem Mühendisliği Yanıtı |
| :--- | :--- |
| **Ne?** | `exec` ailesi; bir sürecin kimliğini (PID/PPID/Creds) koruyarak, sanal adres uzayını, kodunu ve çalışma durumunu yeni bir ELF ikili dosyası ile değiştiren sistem çağrısı mekanizmasıdır. |
| **Neden?** | İşletim sistemlerinde süreç oluşturma (`fork`) ile program çalıştırma (`exec`) birbirinden ayrılmıştır. Bu ayrım, alt sürecin çalıştırılmadan önce dosya yönlendirmelerini (redirection), yetkilerini ve kaynak limitlerini güvenle ayarlamasını sağlar. |
| **Nerede?** | Linux çekirdeğinin `fs/exec.c`, `fs/binfmt_elf.c` altsistemlerinde, POSIX libc katmanında, systemd servis yöneticilerinde ve konteyner motorlarında (runc, crun). |
| **Nasıl?** | Eski sanal bellek sayfalarını `munmap` ile serbest bırakarak, yeni ELF başlıklarını ayrıştırarak, `FD_CLOEXEC` bayraklı tanıtıcıları kapatıp program sayacını (RIP) yeni giriş noktasına (`_start`) atayarak çalışır. |
| **Ne Zaman?** | Bir daemon yeni bir görev başlattığında, kabuk (shell) kullanıcı komutunu çalıştırdığında veya bir alt sistem bağımsız bir ikili dosyaya devredildiğinde. |
| **Kim?** | Askeri görev bilgisayarı yazılımcıları, gömülü Linux mimarları, siber güvenlik araştırmacıları ve sistem çekirdek mühendisleri. |


## 13. Günlük Hayat Benzetmesi: Tiyatro Sahnesinde Aktör ve Rol Değişimi

`exec` zihinsel modelini anlamanın en etkili yolu bir tiyatro sahnesini düşünmektir:

```
+-------------------------------------------------------------------------+
|                              TİYATRO SAHNESİ                            |
|                                                                         |
| Aktör: Ahmet Bey (TC Kimlik No: 4092) ---> [PID KORUNUR: 4092]          |
| Sözleşme: Tiyatro Yönetimi            ---> [PPID KORUNUR: 1001]         |
| Soyunma Odası Anahtarları             ---> [FD Tablosu]                 |
|                                                                         |
| SAHNE 1: "Hamlet" Oyunu (Eski Program)                                  |
|   - Kostüm: Prens Hamlet zırhı         ---> [Eski Kod .text]            |
|   - Replikler: Ezberdeki tiratlar      ---> [Eski Heap/Stack]           |
|                                                                         |
| ======================= YÖNETMEN: "EXECVE!" =========================== |
|                                                                         |
| SAHNE 2: "Othello" Oyunu (Yeni Program)                                 |
|   - Aktör AYNI KİŞİDİR (Ahmet Bey / PID 4092). Yeni bir aktör doğmadı!  |
|   - Kostüm tamamen değiştirildi        ---> [Yeni ELF Kodu .text]       |
|   - Replikler tamamen sıfırlandı       ---> [Yeni Temiz Heap/Stack]     |
|   - Soyunma Odası Anahtarları:                                          |
|     * "Sahne Çıkışında Bırak" etiketli olanlar teslim edildi (CLOEXEC). |
|     * Etiketsiz anahtarlar Ahmet Bey'in cebinde kaldı (FD Sızıntısı!).   |
+-------------------------------------------------------------------------+
```

Bu benzetmede aktörün biyolojik varlığı **süreç kimliğini (PID)**, giydiği kostüm ve oynadığı karakter ise **sanal bellek uzayını (VMA)** temsil eder.


## 14. Neden-Sonuç Tablosu ve Mimari Karar Matrisi

| Karar / Eylem | Doğrudan Sonuç | Potansiyel Risk / Hata Sınıfı | Mimari Tavsiye |
| :--- | :--- | :--- | :--- |
| `O_CLOEXEC` bayrağı kullanmak | Dosya tanıtıcısı `execve` anında çekirdek tarafından otomatik kapatılır. | Hiçbir risk taşımaz; sıfır sızıntı garantisi sağlar. | **ZORUNLU (MISRA / Askeri Standart)** |
| `execve` sonrası normal `exit()` çağırmak | Ebeveynle paylaşılan stdio tamponları çift boşaltılır (double flush). | Log bozulması, telemetri soketinde veri kirliliği. | `_exit(127)` veya `_Exit()` kullanın. |
| Devasa bellekli süreçte `fork()+execve()` | Milyonlarca sayfa tablosu kopyalanır; gecikme ms mertebesine çıkar. | `ENOMEM` (Overcommit reddi), CPU darboğazı. | `posix_spawn(3)` kullanın. |
| `execlp` ile komut çalıştırmak | Komut `$PATH` değişkenindeki dizinlerde aranır. | `$PATH` zehirlenmesi (DLL/Binary Hijacking saldırısı). | Güvenlik kritik kodda tam mutlak yol (`/usr/bin/cmd`) verin. |
| `execle` ile beyaz liste ortamı geçmek | Yeni sürece sadece izin verilen çevre değişkenleri aktarılır. | Gerekli bir kütüphane yolu (`LD_LIBRARY_PATH`) unutulabilir. | Sıkı denetlenmiş minimal ortam dizisi oluşturun. |


## 15. Savunma Sanayii ve Görev Kritik Sistem Senaryoları

### 1. ASELSAN Radar Görev Bilgisayarı: Güvenli Alt Modül Yeniden Başlatma
Radar ana kontrol süreci (Watchdog Daemon), sinyal işleme modülünde (DSP Worker) beklenmeyen bir donma tespit ettiğinde:
- Yeni bir süreç üretmek yerine, mevcut iş parçacığının PID'sini koruyarak `execve("/opt/aselsan/radar_dsp_v2", ...)` çağrısı yapar.
- Bu sayede donanım paylaşımlı bellek dosya tanıtıcısı (`shm_open`) alt sürece aktarılırken, geçici tamponlar ve çöken bellek alanları çekirdek tarafından anında çöpe atılır.

### 2. HAVELSAN Komuta Kontrol (C2) Süreç İzolasyonu ve Sandboxing
C2 sisteminde harici sensörlerden gelen formatlanmamış verileri ayrıştırmak için üçüncü taraf bir ayrıştırıcı çalıştırılacağında:
1. Ana süreç `fork()` çağırır.
2. Çocuk süreçte `setrlimit()` ile CPU ve bellek sınırları daraltılır.
3. `close_range(3, ~0U, 0)` ile tüm kritik soketler kapatılır.
4. `execve("/usr/libexec/untrusted_parser", ...)` ile izole kod fırlatılır.
Bu sayede ayrıştırıcı çökse veya ele geçirilse dahi ana komuta kontrol sistemine zarar veremez.


## 16. Gün Sonu Özeti

1. **`exec` Ailesi Zihinsel Modeli**: `execve` yeni bir süreç başlatmaz; mevcut sürecin PID, PPID ve kimlik bilgilerini koruyarak sanal adres uzayını, kodunu, verisini ve yığınını tamamen yeni bir ELF ikili dosyası ile baştan aşağı yeniler.
2. **Çözdüğü Problem ve Avantajı**: Süreç oluşturma (`fork`) ile program yürütme (`exec`) adımlarını ayrıştırarak; alt sürecin çalışmadan önce dosya yönlendirmelerini, yetki düşürmelerini ve çevre değişkenlerini güvenle yapılandırmasına olanak tanır.
3. **Getirdiği Risk ve Hata Sınıfı**: `FD_CLOEXEC` bayrağı unutulduğunda kritik dosya tanıtıcıları yeni programa sızarak ciddi güvenlik açıklarına (CVE-2019-5736 benzeri) yol açar; ayrıca büyük süreçlerde `fork()+execve()` sayfa tablosu kopyalama yükü nedeniyle ciddi gecikmelere sebep olur (`posix_spawn` alternatifi gerektirir).


## 17. Gelecek Gün Köprüsü: Gün 088

Yarın **GÜN 088** kapsamında:
- **Konu**: **Exit Code ve Process Lifecycle** (Süreç Yaşam Döngüsü ve Çıkış Durumları)
- **Kapsam**: `exit(3)` vs `_exit(2)`, `WEXITSTATUS`, `WIFSIGNALED`, `WTERMSIG`, Core Dump mekanizmaları, Zombi (`<defunct>`) ve Yetim (Orphan) süreçlerin çekirdek düzeyinde sonlandırılması.
- **Hazırlık**: Bir alt sürecin `execve` sonrasında ürettiği çıkış kodunun ebeveyn tarafından nasıl yorumlandığını inceleyerek süreç yaşam döngüsü mimarisini tamamlayacağız.
